In [1]:
import pandas as pd

base = "/root/autodl-tmp/DATA/Descriptive_Statistics"

set1 = pd.read_parquet(f"{base}/set1_main_grf.parquet")
set2 = pd.read_parquet(f"{base}/set2_main_grf.parquet")
set3 = pd.read_parquet(f"{base}/set3_main_grf.parquet")

print("set1 shape:", set1.shape)
print("set2 shape:", set2.shape)
print("set3 shape:", set3.shape)

set1 shape: (4074644, 723)
set2 shape: (4074644, 723)
set3 shape: (4074644, 723)


In [14]:
selected_vars = [
    "GDP_growth",
    "Cash Dividend per Share before Taxes",
    "Cash Paid to and on Behalf of Employees",
    "Cash Dividend per Share after Taxes",
    "Beginning Balance of Cash and Cash Equivalents",
    "Ratio of financial liabilities",
    "Earning Retention Ratio",
    "bankd",
    "Ratio of Operating liabilities",
    "AQsp_time_lag1",
    "AQsp_Volume_lag1",
    "AQsp_Amount_lag1"
]

# 对这些变量的缺失值填补为 0
set1[selected_vars] = set1[selected_vars].fillna(0)
set2[selected_vars] = set2[selected_vars].fillna(0)
set3[selected_vars] = set3[selected_vars].fillna(0)

In [3]:
from scipy.stats import kurtosis  # 峰度计算的核心依赖，必须导入
def summary_stats(df, vars_list):
    X = df[vars_list]

    out = pd.DataFrame(index=vars_list)
    out["N"] = X.notna().sum()
    out["Mean"] = X.mean(numeric_only=True)
    out["Median"] = X.median(numeric_only=True)
    out["Std"] = X.std(numeric_only=True)

    # 分位数：你也可以改成 0.05/0.95 等
    q = X.quantile([0.01, 0.25, 0.75, 0.99], numeric_only=True).T
    q.columns = ["P1", "P25", "P75", "P99"]
    out = out.join(q)

    # 峰度：fisher=False 为“原始峰度”（常见），fisher=True 为“超额峰度”
    out["Kurtosis"] = X.apply(
        lambda s: kurtosis(s.dropna().values, fisher=False) if s.dropna().shape[0] > 3 else np.nan
    )

    return out.reset_index().rename(columns={"index": "Variable"})

def build_three_panels(df, y_col, vars_list):
    panelA = summary_stats(df, vars_list)
    panelB = summary_stats(df[df[y_col] == 1], vars_list)
    panelC = summary_stats(df[df[y_col] == 0], vars_list)
    return panelA, panelB, panelC

In [11]:
import numpy as np
import pandas as pd

# 直接定义你需要的十个变量名
scale_vars = [
    "Cash Paid to and on Behalf of Employees",
    "Beginning Balance of Cash and Cash Equivalents"
]
scale_factor = 1e-8  # 10^-8

# 定义需要处理的数据集列表
dataset_list = ["set1", "set2", "set3"]

# 循环处理每个数据集
for set_name in dataset_list:
    # 跳过不存在的数据集（可选，避免KeyError）
    if set_name not in all_data:
        print(f"Warning: {set_name} does not exist in all_data, skipped.")
        continue
    
    # 筛选出当前数据集中实际存在的变量
    vars_to_output = [c for c in selected_vars if c in all_data[set_name].columns]

    # 对存在于当前数据集中的缩放变量进行缩放（乘以10^-8）
    scale_vars_exist = [c for c in scale_vars if c in all_data[set_name].columns]
    if scale_vars_exist:
        for c in scale_vars_exist:
            # 防止该列是object/string导致无法乘法：强制转为数值，无法转换的设为NaN
            all_data[set_name][c] = pd.to_numeric(all_data[set_name][c], errors="coerce") * scale_factor

        # 仅保留关键操作结果打印
        print(f"[{set_name}] 完成处理")
    else:
        # 仅保留必要的警告提示
        print(f"[{set_name}] 完成处理")

[set1] 完成处理
[set2] 完成处理
[set3] 完成处理


In [15]:
# 定义需要处理的数据集列表
dataset_list = ["set1", "set2", "set3"]

all_data = {
    "set1": set1,
    "set2": set2,
    "set3": set3
}

vars_to_output = [c for c in selected_vars if c in all_data[set_name].columns]

results = {}
for set_name in ["set1","set2","set3"]:
    df = all_data[set_name]

    y_col = "insider_trading"

    numeric_vars = [c for c in vars_to_output if pd.api.types.is_numeric_dtype(df[c])]

    print(f"\n[{set_name}] variables used:", numeric_vars)

    panelA, panelB, panelC = build_three_panels(df, y_col, numeric_vars)

    results[set_name] = {"A": panelA, "B": panelB, "C": panelC}


[set1] variables used: ['GDP_growth', 'Cash Dividend per Share before Taxes', 'Cash Paid to and on Behalf of Employees', 'Cash Dividend per Share after Taxes', 'Beginning Balance of Cash and Cash Equivalents', 'Ratio of financial liabilities', 'Earning Retention Ratio', 'bankd', 'Ratio of Operating liabilities', 'AQsp_time_lag1', 'AQsp_Volume_lag1', 'AQsp_Amount_lag1']

[set2] variables used: ['GDP_growth', 'Cash Dividend per Share before Taxes', 'Cash Paid to and on Behalf of Employees', 'Cash Dividend per Share after Taxes', 'Beginning Balance of Cash and Cash Equivalents', 'Ratio of financial liabilities', 'Earning Retention Ratio', 'bankd', 'Ratio of Operating liabilities', 'AQsp_time_lag1', 'AQsp_Volume_lag1', 'AQsp_Amount_lag1']

[set3] variables used: ['GDP_growth', 'Cash Dividend per Share before Taxes', 'Cash Paid to and on Behalf of Employees', 'Cash Dividend per Share after Taxes', 'Beginning Balance of Cash and Cash Equivalents', 'Ratio of financial liabilities', 'Earning 

In [16]:
out_xlsx = "/root/autodl-tmp/Table1_and_AppendixTableA1/summary_selected_vars_with_grf.xlsx"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    for set_name, panels in results.items():
        panels["A"].to_excel(writer, sheet_name=f"{set_name}_PanelA", index=False, float_format="%.3f")
        panels["B"].to_excel(writer, sheet_name=f"{set_name}_PanelB", index=False, float_format="%.3f")
        panels["C"].to_excel(writer, sheet_name=f"{set_name}_PanelC", index=False, float_format="%.3f")

print("Saved:", out_xlsx)

Saved: /root/autodl-tmp/Table1/summary_selected_vars_with_grf.xlsx
